In [4]:
# Requirements: pandas, numpy
# Usage: set IN_PATH and OUT_PATH, then run.

import pandas as pd
import numpy as np
from pathlib import Path

# ---- Configuration (edit as needed)
IN_PATH  = Path("data/iam_raw.csv")                 # your input file
OUT_PATH = Path("electricity_mix_2025_2100.csv")
REGION_FILTER   = "World"                      # e.g., "World"
SCENARIO_FILTER = None                         # e.g., "SSP2-26" or None to keep all
MODEL_FILTER    = None                         # e.g., "IMAGE 3.0.1" or None to keep all
VARIABLE_PREFIX = "Secondary Energy|Electricity|"  # electricity variables in IAMC format
TARGET_YEARS = np.arange(2025, 2101)           # 2025–2100 inclusive

# ---- Load
df = pd.read_csv(IN_PATH)

# Basic cleaning: keep only rows that look like real data
df = df.dropna(subset=["Variable", "Region"])

# Optional filters
if REGION_FILTER is not None:
    df = df[df["Region"].astype(str).str.strip().eq(REGION_FILTER)]
if SCENARIO_FILTER is not None:
    df = df[df["Scenario"].astype(str).str.strip().eq(SCENARIO_FILTER)]
if MODEL_FILTER is not None:
    df = df[df["Model"].astype(str).str.strip().eq(MODEL_FILTER)]

# Keep electricity-related variables
elec = df[df["Variable"].astype(str).str.startswith(VARIABLE_PREFIX)].copy()
if elec.empty:
    raise ValueError("No electricity rows found. Check VARIABLE_PREFIX/filters.")

# Extract source name after the last '|'
elec["Source"] = elec["Variable"].astype(str).str.split("|").str[-1].str.strip()

# Identify numeric year columns (IAMC wide format)
year_cols = [c for c in elec.columns if str(c).isdigit()]
if not year_cols:
    raise ValueError("No year columns detected (e.g., '2020', '2030', ...).")

# Melt to long format
melt = elec.melt(
    id_vars=["Model", "Scenario", "Region", "Source", "Unit"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Value",
)
melt["Year"] = pd.to_numeric(melt["Year"], errors="coerce")
melt["Value"] = pd.to_numeric(melt["Value"], errors="coerce")
melt = melt.dropna(subset=["Year", "Source", "Value"])

# Aggregate duplicates if any (same Model/Scenario/Region/Source/Year)
melt = (melt
        .groupby(["Model", "Scenario", "Region", "Source", "Unit", "Year"], as_index=False)["Value"]
        .sum())

# Compute shares (%) within each (Model, Scenario, Region, Year)
melt["Total"] = melt.groupby(["Model", "Scenario", "Region", "Year"])["Value"].transform("sum")
melt["Share_pct"] = (melt["Value"] / melt["Total"].replace(0, np.nan)) * 100.0

# Build annual interpolation 2025–2100 for each (Model, Scenario, Region)
out_frames = []
for (model, scen, region), sub in melt.groupby(["Model", "Scenario", "Region"], dropna=False):
    wide = sub.pivot(index="Year", columns="Source", values="Share_pct").sort_index()
    # Insert missing target years and interpolate linearly along the index
    years_union = sorted(set(wide.index).union(TARGET_YEARS))
    wide_interp = (wide
                   .reindex(years_union)
                   .interpolate(method="index", limit_direction="both")
                   .loc[TARGET_YEARS])

    # Renormalize rows to exactly 100% (protect against small drift)
    wide_interp = wide_interp.div(wide_interp.sum(axis=1), axis=0) * 100.0
    wide_interp = wide_interp.round(4)

    # Attach identifiers
    wide_interp.insert(0, "Year", TARGET_YEARS)
    wide_interp.insert(1, "Model", model)
    wide_interp.insert(2, "Scenario", scen)
    wide_interp.insert(3, "Region", region)
    out_frames.append(wide_interp.reset_index(drop=True))

result = pd.concat(out_frames, ignore_index=True)

# If there’s only one (Model, Scenario, Region), drop those cols for a cleaner file
if result[["Model","Scenario","Region"]].drop_duplicates().shape[0] == 1:
    result = result.drop(columns=["Model","Scenario","Region"])

# Save
result.to_csv(OUT_PATH, index=False)
print(f"Wrote: {OUT_PATH.resolve()}")


Wrote: /Users/farshidnazemi/pack-repo/packaging-roadmap/electricity_mix_2025_2100.csv
